# Week 3 · Exercise: GBMs, SHAP and neural-network basics

**Goal:** (1) see whether XGBoost / LightGBM beat the Week 2 baseline, (2) explain a boosted model
with SHAP, (3) learn how a neural network works by building one in PyTorch, and compare it with the
trees on the same fraud data.

Data and protocol are Week 2's: `data/raw/creditcard.csv` (see *Reproducing this* in
`notes/week-02.md`), a stratified 70/15/15 split, fit on `train`, compare on `val`. The `test` split is
never scored here. Look for **✏️ Try it** cells. The slowest cell (training the network) takes about a minute.

Finished code: `classical_ml/reports/week3_gbm.py` (5-fold comparison) and `week3_nn.py`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from classical_ml.data import train_val_test_split

df = pd.read_csv("../../data/raw/creditcard.csv")
train, val, test = train_val_test_split(df, target="Class")
Xtr, ytr = train.drop(columns="Class"), train["Class"]
Xv, yv = val.drop(columns="Class"), val["Class"]
print(f"train {len(train):,} ({ytr.sum()} frauds) | val {len(val):,} ({yv.sum()} frauds)")

train 199,364 (344 frauds) | val 42,721 (74 frauds)


## 1. Gradient-boosting libraries: same idea, three programs

All three build many small decision trees in sequence, each fixing the errors left by the ones before
(Week 2, section 10). They differ in speed, how they grow trees and their defaults.

| | who | distinctive |
|---|---|---|
| **HistGradientBoosting** | scikit-learn | built in, histogram-based (features sorted into ~256 buckets) |
| **XGBoost** | XGBoost project | long-time standard, rich regularization |
| **LightGBM** | Microsoft | histogram-based, grows trees *leaf-wise* (splits the leaf that cuts error most) |

Settings used (hand-picked, **not tuned**): 300 trees, learning rate 0.05, L2 penalty 10, minimum leaf
sizes, 80% row and feature sampling for XGBoost and LightGBM. The names differ per library:
`l2_regularization` / `reg_lambda`, `min_samples_leaf` / `min_child_weight` / `min_child_samples`,
`max_iter` / `n_estimators`.

In [4]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=63, min_child_samples=50,
    reg_lambda=10.0, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    n_jobs=-1, random_state=0, verbose=-1,
).fit(Xtr, ytr)
p_lgbm = lgbm.predict_proba(Xv)[:, 1]
print(f"LightGBM validation PR-AUC: {average_precision_score(yv, p_lgbm):.3f}")

LightGBM validation PR-AUC: 0.826


The 5-fold comparison (same folds and same 492 frauds as Week 2) takes several minutes, so it lives
in a script: `uv run python classical_ml/reports/week3_gbm.py`. Its table:

In [5]:
from pathlib import Path

from IPython.display import Markdown, display

path = Path("../reports/week3_gbm.md")
display(Markdown(path.read_text(encoding="utf-8") if path.exists() else "*run the script above first*"))

| Model | PR-AUC mean ± std | per fold | vs Week 2 baseline (mean diff ± std, folds better) |
|---|---|---|---|
| XGBoost | 0.858 ± 0.025 | 0.835 0.896 0.877 0.850 0.830 | -0.001 ± 0.006, 3/5 |
| LightGBM | 0.862 ± 0.022 | 0.853 0.897 0.875 0.851 0.833 | +0.003 ± 0.001, 5/5 |
| HistGB (Week 2) | 0.859 ± 0.022 | 0.848 0.895 0.871 0.849 0.830 | -0.000 ± 0.000, 2/5 |


**What it shows:** XGBoost and LightGBM land within about 0.004 PR-AUC of the Week 2 baseline. LightGBM
won all 5 folds but by only +0.003, a tenth of the fold-to-fold spread. Same algorithm, same
regularization, same features, so the same answer. The *family* of model (trees vs straight-line)
mattered 30 times more than the *library*.

**✏️ Try it:** change `num_leaves` from 15 to 63 and rerun the cell above. Does the validation score
move by more than ±0.02, the noise level from 74 frauds?

In [ ]:
# your code here

## 2. SHAP: why did the model flag this transaction?

SHAP gives every feature credit or blame for moving **one transaction's** score away from the model's
average score. The pieces add up exactly to the score, in *log-odds* (a stretched probability).

In [6]:
import shap

explainer = shap.TreeExplainer(lgbm)
sv = explainer.shap_values(Xv)
sv = sv[1] if isinstance(sv, list) else sv          # (rows, features), log-odds
base = float(np.ravel(explainer.expected_value)[-1])

imp = pd.Series(np.abs(sv).mean(axis=0), index=Xv.columns).sort_values(ascending=False)
print("global importance = average |SHAP| over all validation rows")
print(imp.head(8).round(3).to_string())
print(f"\ntop 5 features carry {imp.head(5).sum() / imp.sum():.0%} of the total: the signal is spread out")

D:\Coding and AI\AI Engineer Roadmap\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


global importance = average |SHAP| over all validation rows
V4     0.753
V14    0.319
V12    0.227
V8     0.191
V26    0.149
V15    0.140
V11    0.134
V16    0.120

top 5 features carry 44% of the total: the signal is spread out


D:\Coding and AI\AI Engineer Roadmap\.venv\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


**Global:** `V4` matters most, at twice the next feature. `Time` and `Amount` rank 6th and 8th: the
model uses them, consistent with Week 2's engineered versions adding nothing.

**Local:** take one fraud (the median-scored one) and see which features pushed its score up.

In [7]:
fraud_rows = np.where(yv.to_numpy() == 1)[0]
i = fraud_rows[np.argsort(-p_lgbm[fraud_rows])[len(fraud_rows) // 2]]

s = pd.Series(sv[i], index=Xv.columns)
top = s.reindex(s.abs().sort_values(ascending=False).index).head(5)
print(f"P(fraud) = {p_lgbm[i]:.3f}")
print(f"log-odds: base {base:.2f} + sum of pushes {s.sum():+.2f} = {base + s.sum():.2f}")
pd.DataFrame({"value": Xv.iloc[i][top.index].round(2), "push (log-odds)": top.round(2)})

P(fraud) = 0.941
log-odds: base -10.55 + sum of pushes +13.32 = 2.76


,value,push (log-odds)
V14,-8.51,5.09
V10,-4.98,2.20
V12,-6.41,1.63
V7,-3.03,0.74
V3,-3.75,0.60


Several features had extreme values at once, and each pushed the score up. Cautions:

- SHAP shows what the model **relies on**, not what **causes** fraud. Correlated features share credit.
- With anonymized `V1`-`V28`, "V14 was very negative" ranks the evidence but isn't a reason you could
  tell a customer or regulator. With named features it would be.
- The base value (about -10) is the average *raw score*, dominated by the 99.8% normal rows. It is not a
  fraud rate.

**✏️ Try it:** explain the *highest-scoring normal* transaction (a false alarm): which features made the
model suspicious? Use `np.where(yv.to_numpy() == 0)[0]` and `p_lgbm`.

In [ ]:
# your code here

## 3. Neural networks from the ground up

### 3.1 A tensor
A **tensor** is an array of numbers that PyTorch can track calculations on. Same idea as a NumPy array.

In [8]:
import torch
from torch import nn

x = torch.tensor([2.0, 3.0])
print(x, x.dtype, x.shape)

tensor([2., 3.]) torch.float32 torch.Size([2])


### 3.2 One neuron
A neuron computes `z = w1*x1 + w2*x2 + b`, then bends it with an **activation function**. The weights
`w` and bias `b` are what the network learns.

- **ReLU**: `max(0, z)`. Anything negative becomes 0.
- **Sigmoid**: squashes `z` to between 0 and 1, so it can be read as a probability.

In [9]:
w = torch.tensor([0.5, -1.0])
b = torch.tensor(0.1)
z = (w * x).sum() + b                      # 0.5*2 + (-1)*3 + 0.1
print(f"z = {z.item():.2f}")
print(f"ReLU(z)    = {torch.relu(z).item():.2f}")
print(f"sigmoid(z) = {torch.sigmoid(z).item():.3f}")

z = -1.90
ReLU(z)    = 0.00
sigmoid(z) = 0.130


### 3.3 Why the bend matters
Without an activation, stacking layers gains nothing: two linear layers are mathematically **one**
linear layer. Check it:

In [10]:
torch.manual_seed(0)
l1, l2 = nn.Linear(3, 4), nn.Linear(4, 2)
inp = torch.randn(5, 3)

two_layers = l2(l1(inp))                                    # no activation between them
W = l2.weight @ l1.weight                                   # one equivalent weight matrix
bias = l2.weight @ l1.bias + l2.bias
one_layer = inp @ W.T + bias
print("two linear layers == one linear layer:", torch.allclose(two_layers, one_layer, atol=1e-6))

two linear layers == one linear layer: True


Put a ReLU between them and that shortcut no longer exists: the network can bend, so it can represent
curves and combinations of features. That is the whole reason activations exist.

### 3.4 Loss: how wrong is the prediction?
For a yes/no target we use **binary cross-entropy**. For a true fraud (`y = 1`), a prediction `p` costs
`-ln(p)`:

In [11]:
for p in (0.99, 0.9, 0.5, 0.1, 0.01):
    print(f"true fraud, predicted P(fraud) = {p:<4}  ->  loss {-np.log(p):.3f}")

true fraud, predicted P(fraud) = 0.99  ->  loss 0.010
true fraud, predicted P(fraud) = 0.9   ->  loss 0.105
true fraud, predicted P(fraud) = 0.5   ->  loss 0.693
true fraud, predicted P(fraud) = 0.1   ->  loss 2.303
true fraud, predicted P(fraud) = 0.01  ->  loss 4.605


A confident correct call is cheap and a confident **wrong** call is very expensive, which pushes the
model to be right, and to be honest about uncertainty.

**✏️ Try it:** what does a *normal* transaction (`y = 0`) cost when the model says P(fraud) = 0.9?
(The loss for `y = 0` is `-ln(1 - p)`.)

In [17]:
p = 0.9
print(-np.log(1 - p))      # loss for a normal transaction (y = 0)

2.302585092994046


### 3.5 Gradients and autograd
The **gradient** of the loss with respect to a weight answers: "if I raise this weight a little, does
the loss go up or down, and how fast?" **Autograd** computes it automatically.

Fit one weight `w` so that `w * x` matches the target `y`:

In [18]:
xs = torch.tensor([2.0, 3.0])
ys = torch.tensor([6.0, 9.0])                 # the right answer is w = 3
w = torch.tensor(1.5, requires_grad=True)

loss = ((w * xs - ys) ** 2).mean()
loss.backward()                               # autograd fills in w.grad
print(f"loss {loss.item():.3f}, slope d loss / d w = {w.grad.item():.2f}")

by_hand = (2 * (1.5 * xs - ys) * xs).mean()   # the calculus, written out
print(f"by hand: {by_hand.item():.2f}   (negative slope: raising w lowers the loss)")

loss 14.625, slope d loss / d w = -19.50
by hand: -19.50   (negative slope: raising w lowers the loss)


### 3.6 Gradient descent: nudge the weight against its slope, repeat
Each step: `w = w - learning_rate * slope`. The **learning rate** is the step size.

In [19]:
w = torch.tensor(1.5, requires_grad=True)
lr = 0.05
for step in range(1, 21):
    loss = ((w * xs - ys) ** 2).mean()
    w.grad = None
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
    if step in (1, 2, 5, 10, 20):
        print(f"step {step:2d}: w = {w.item():.3f}, loss = {loss.item():.4f}")

step  1: w = 2.475, loss = 14.6250
step  2: w = 2.816, loss = 1.7916
step  5: w = 2.992, loss = 0.0033
step 10: w = 3.000, loss = 0.0000
step 20: w = 3.000, loss = 0.0000


`w` walks from 1.5 toward the correct 3 and the loss shrinks. A real network does exactly this for
thousands of weights at once.

**✏️ Try it:** set `lr = 0.5` and rerun. What happens, and why? Then try `lr = 0.001`. (Too big a step
overshoots; too small crawls.)

In [20]:
# your code here

### 3.7 A network and its training loop
A **layer** is many neurons side by side. Our fraud network: 30 inputs, then 64 neurons, 32 neurons,
and 1 output. Count the weights: `30*64+64` + `64*32+32` + `32*1+1` = **4,097**.

In [21]:
def make_model():
    return nn.Sequential(nn.Linear(30, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

print(make_model())
print("parameters:", sum(p.numel() for p in make_model().parameters()))

Sequential(
  (0): Linear(in_features=30, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)
parameters: 4097


Networks are far more sensitive to feature **scale** than trees (trees only care about the *order* of
values). So scale first, fitting the scaler on **train only** (Week 1's leakage rule).

Then every training loop has the same four lines:

```
opt.zero_grad()                 # clear old slopes
loss = loss_fn(model(x), y)     # forward pass: predict, measure the error
loss.backward()                 # autograd: slope of the loss for every weight
opt.step()                      # Adam moves each weight a little against its slope
```

An **epoch** is one pass over the training data. A **batch** (here 1,024 rows) is the chunk used for each
nudge. **Adam** is gradient descent that adapts its step size for each weight.

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(Xtr)
x_tr = torch.tensor(scaler.transform(Xtr), dtype=torch.float32)
y_tr = torch.tensor(ytr.to_numpy(), dtype=torch.float32)
x_v = torch.tensor(scaler.transform(Xv), dtype=torch.float32)

def train_nn(epochs=30, batch=1024, pos_weight=None, seed=0, report=(1, 2, 5, 10, 20, 30)):
    torch.manual_seed(seed)
    model = make_model()
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=None if pos_weight is None else torch.tensor(float(pos_weight))
    )
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(x_tr))
        total = 0.0
        for i in range(0, len(order), batch):
            idx = order[i : i + batch]
            opt.zero_grad()
            loss = loss_fn(model(x_tr[idx]).squeeze(1), y_tr[idx])
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            p = torch.sigmoid(model(x_v).squeeze(1)).numpy()
        history.append((epoch, total / len(x_tr), average_precision_score(yv, p)))
        if epoch in report:
            print(f"epoch {epoch:2d}: train loss {history[-1][1]:.5f} | val PR-AUC {history[-1][2]:.3f}")
    return model, p, history

model, p_nn, hist = train_nn()
best = max(hist, key=lambda h: h[2])
print(f"best val PR-AUC {best[2]:.3f} at epoch {best[0]} (picked on val, so optimistic)")
print(f"LightGBM on the same split: {average_precision_score(yv, p_lgbm):.3f}")

epoch  1: train loss 0.12476 | val PR-AUC 0.487
epoch  2: train loss 0.00563 | val PR-AUC 0.562
epoch  5: train loss 0.00318 | val PR-AUC 0.643
epoch 10: train loss 0.00225 | val PR-AUC 0.617
epoch 20: train loss 0.00154 | val PR-AUC 0.759
epoch 30: train loss 0.00107 | val PR-AUC 0.771
best val PR-AUC 0.781 at epoch 24 (picked on val, so optimistic)
LightGBM on the same split: 0.826


How to read it:

- **Training loss keeps falling**, but validation PR-AUC is bumpy (it can dip between epochs). With
  ~0.17% fraud, each batch of 1,024 rows holds only a couple of frauds, so the updates are noisy.
- **The network beats plain logistic regression** (0.67) because the hidden layers add the bends from
  3.3, but it **doesn't beat LightGBM** on this table. Trees need no scaling, handle sharp rules and
  interactions with little tuning, and train in seconds. The network has only 344 training frauds to
  learn from.
- **This is evidence, not proof.** One split, one seed and 74 frauds: a 0.05 gap is within the noise from
  Week 2. Cross-validation over several seeds would settle it.
- The best epoch was chosen by looking at validation, the same trick as tuning a threshold on it, so
  treat it as optimistic.

## ✏️ Your turn

Do these here first, then move anything reusable into `classical_ml/reports/` **with a test**.

1. **Seeds.** Run `train_nn(seed=s)` for `s` in 0..4 and take the final-epoch PR-AUC of each. How much
   do they differ from each other? Is that spread bigger than the gap to LightGBM?
2. **Class weight.** `pos_weight=50` makes each fraud count 50 times as much in the loss. Does PR-AUC
   improve, or does it behave like Week 2's class weights (more flagged, same ranking)?
3. **Size.** Change `make_model` to hidden layers of 8 and 4 (set the first Linear to `nn.Linear(30, 8)`
   etc.). How few weights can you get away with?
4. **Overfitting.** Train for 100 epochs. Does training loss keep falling while validation turns down?
   Where would early stopping have stopped?
5. **No scaling.** Skip `StandardScaler` (use raw `Xtr` values). What happens to the loss and the score,
   and why does `Amount` (up to 25,000) cause it?
6. **Write-up.** Fill in the *What confused me* part of `notes/week-03.md` and answer in your own words:
   "Why do gradient-boosted trees often beat neural networks on tabular data?"